# Rocket Engine Performance Prediction: Rediscovering the Isp = f(sqrt(Tc/M)) Law

## **Table of Contents**

- [1 - Business Objective](#1-business-objective)
  - [1.1 - Overview](#11-overview)
  - [1.2 - Business Objective Statement](#12-business-objective-statement)
- [2 - Problem Statement](#2-problem-statement)
  - [2.1 - Overview](#21-overview)
- [3 - Solution Methodology](#3-solution-methodology)
  - [3.1 - Overview](#31-overview)
  - [3.2 - The Rediscovery Standard](#32-the-rediscovery-standard)
- [4 - A Brief History: From Hydrazine to Methalox](#4-a-brief-history-from-hydrazine-to-methalox)
  - [4.1 - Overview](#41-overview)
  - [4.2 - Historical Engines and Propellant Combinations](#42-historical-engines-and-propellant-combinations)
- [5 - The Science: Rocket Propulsion Fundamentals](#5-the-science-rocket-propulsion-fundamentals)
  - [5.1 - The Tsiolkovsky Rocket Equation](#51-the-tsiolkovsky-rocket-equation)
  - [5.2 - The Ideal Rocket Nozzle Equation](#52-the-ideal-rocket-nozzle-equation)
  - [5.3 - Area Ratio and the Pressure Ratio Relationship](#53-area-ratio-and-the-pressure-ratio-relationship)
  - [5.4 - Characteristic Velocity and Thrust Coefficient](#54-characteristic-velocity-and-thrust-coefficient)
  - [5.5 - Combustion Efficiency and Why Real Engines Underperform the Ideal Equation](#55-combustion-efficiency-and-why-real-engines-underperform-the-ideal-equation)
- [6 - Installing and Importing the Libraries](#6-installing-and-importing-the-libraries)
  - [6.1 - Overview](#61-overview)
- [7 - Generating the Synthetic Rocket Engine Dataset](#7-generating-the-synthetic-rocket-engine-dataset)
  - [7.1 - Overview](#71-overview)
  - [7.2 - Propellant Parameter Table](#72-propellant-parameter-table)
  - [7.3 - Sanity Check Against Published Performance](#73-sanity-check-against-published-performance)
- [8 - Exploratory Data Analysis and Human-Engineered Features](#8-exploratory-data-analysis-and-human-engineered-features)
  - [8.1 - Overview](#81-overview)
  - [8.2 - Human-Engineered Feature 1 and 2: Pressure Ratio and Expansion Ratio](#82-human-engineered-feature-1-and-2-pressure-ratio-and-expansion-ratio)
  - [8.3 - Human-Engineered Feature 3: Characteristic Velocity (c*)](#83-human-engineered-feature-3-characteristic-velocity-c)
  - [8.4 - Human-Engineered Feature 4: Thrust Coefficient (Cf)](#84-human-engineered-feature-4-thrust-coefficient-cf)
  - [8.5 - Human-Engineered Feature 5: Mixture-Ratio Stoichiometry Deviation](#85-human-engineered-feature-5-mixture-ratio-stoichiometry-deviation)
- [9 - GenAI-Generated Features](#9-genai-generated-features)
  - [9.1 - Overview](#91-overview)
  - [9.2 - Constructing the AI-Suggested Features](#92-constructing-the-ai-suggested-features)
  - [9.3 - Ablation: Human Features vs. AI Features vs. Combined](#93-ablation-human-features-vs-ai-features-vs-combined)
- [10 - GenAI Synthetic Data Augmentation](#10-genai-synthetic-data-augmentation)
  - [10.1 - Overview](#101-overview)
  - [10.2 - Sampling Synthetic Rows from the LLM-Proposed Distributions](#102-sampling-synthetic-rows-from-the-llm-proposed-distributions)
  - [10.3 - Three-Way Generalization Check](#103-three-way-generalization-check)
- [11 - Classical ML Model: Gradient Boosting and Random Forest](#11-classical-ml-model-gradient-boosting-and-random-forest)
  - [11.1 - Overview](#111-overview)
  - [11.2 - Selecting the Reference Classical Model](#112-selecting-the-reference-classical-model)
- [12 - Light Deep Learning Model: PyTorch Feedforward Network](#12-light-deep-learning-model-pytorch-feedforward-network)
  - [12.1 - Overview](#121-overview)
  - [12.2 - Comparing the Neural Network to the Classical Model](#122-comparing-the-neural-network-to-the-classical-model)
- [13 - Foundation Model Benchmark: TabPFN](#13-foundation-model-benchmark-tabpfn)
  - [13.1 - Overview](#131-overview)
  - [13.2 - Installing and Fitting TabPFN](#132-installing-and-fitting-tabpfn)
  - [13.3 - Three-Way Model Comparison](#133-three-way-model-comparison)
- [14 - Explainability: Permutation Importance and Partial Dependence](#14-explainability-permutation-importance-and-partial-dependence)
  - [14.1 - Why Not SHAP for This Case](#141-why-not-shap-for-this-case)
  - [14.2 - Permutation Importance](#142-permutation-importance)
  - [14.3 - Partial Dependence Plots](#143-partial-dependence-plots)
- [15 - The Law Rediscovery Moment: Isp Scales as sqrt(Tc / M)](#15-the-law-rediscovery-moment-isp-scales-as-sqrttc-m)
  - [15.1 - What the Explanation Surfaces](#151-what-the-explanation-surfaces)
  - [15.2 - Recovering the Ratio Post-Hoc](#152-recovering-the-ratio-post-hoc)
  - [15.3 - The Historical Payoff](#153-the-historical-payoff)
- [16 - Agentic Layer: LangGraph Design Recommendation](#16-agentic-layer-langgraph-design-recommendation)
  - [16.1 - Overview](#161-overview)
  - [16.2 - Running the Agent on a Sample Design](#162-running-the-agent-on-a-sample-design)
  - [16.3 - From Fixed Pipeline to Autonomous Agent](#163-from-fixed-pipeline-to-autonomous-agent)
  - [16.4 - Tools, Memory, and the ReAct Graph](#164-tools-memory-and-the-react-graph)
  - [16.5 - Running the Autonomous Agent on Sample Designs](#165-running-the-autonomous-agent-on-sample-designs)
- [17 - Interactive Prediction Demo](#17-interactive-prediction-demo)
  - [17.1 - Overview](#171-overview)
- [18 - Conclusion and Takeaways](#18-conclusion-and-takeaways)
  - [18.1 - Conclusion](#181-conclusion)
  - [18.2 - Takeaways](#182-takeaways)

## **1 - Business Objective**

### **1.1 - Overview**

A liquid bipropellant rocket engine converts chemical energy stored in a fuel and an oxidizer into
directed kinetic energy of an exhaust jet. The design choices that govern this conversion, chamber
pressure, propellant combination, mixture ratio, and nozzle expansion ratio, are fixed early in a
program and are expensive to revisit once hardware exists. A propulsion team that can estimate
specific impulse (Isp) from these design parameters before committing to a test campaign saves
months of iteration and millions of dollars in hot-fire testing.

Isp measures propellant efficiency: the thrust produced per unit weight of propellant consumed per
second. A higher Isp means a vehicle reaches a given delta-v with less propellant mass, which is the
single largest lever on payload capacity for any launch vehicle. Every propellant combination and
nozzle geometry decision in the history of rocketry traces back to this one number.

The business objective of this case study is to build a model that predicts Isp from engine design
parameters, so that a design team can screen thousands of candidate configurations computationally
before selecting a small number for physical testing. The secondary objective is explainability: the
model must expose which design parameters actually drive Isp, so that the screening tool teaches the
same physical intuition that historically drove propulsion engineers away from hydrocarbon fuels and
toward liquid hydrogen for missions where Isp dominates the trade.

### **1.2 - Business Objective Statement**

Given a set of engine design parameters (chamber pressure, expansion ratio, propellant combination,
mixture ratio, chamber temperature, and exhaust molecular weight), predict the specific impulse of a
liquid bipropellant rocket engine, and identify which of those parameters the model relies on most
so that the result can be checked against known propulsion theory.

## **2 - Problem Statement**

### **2.1 - Overview**

This is a supervised regression problem. The target variable is specific impulse, Isp, measured in
seconds. The input variables are:

| Variable | Symbol | Typical range |
|---|---|---|
| Chamber pressure | Pc | 5 to 25 MPa |
| Expansion ratio | epsilon = Ae / At | 10 to 200 |
| Mixture ratio (oxidizer to fuel mass ratio) | O/F | propellant-dependent |
| Chamber temperature | Tc | 2500 to 3700 K |
| Exhaust molecular weight | M | 13 to 27 kg/kmol |
| Ratio of specific heats | gamma | 1.14 to 1.24 |
| Propellant combination | (categorical) | LOX/RP-1, LOX/LH2, N2O4/UDMH, LOX/CH4 |

No public dataset of proprietary engine test data at this granularity is available outside engine
manufacturers, so this case study generates a synthetic dataset from the governing equations of
rocket propulsion (Section 5), with realistic parameter ranges drawn from published engine
performance data. The synthetic generation is described in full in Section 7, including the noise
model used to represent combustion efficiency losses that any real engine exhibits relative to the
ideal, loss-free equations.

The evaluation metric is R-squared and root mean squared error (RMSE) on Isp in seconds, computed on
a held-out test split. A propulsion engineer would consider a model useful for early screening if it
predicts Isp within a few seconds of the physics-based ground truth, well inside the noise band
introduced by manufacturing tolerances and combustion inefficiency in a real engine.

## **3 - Solution Methodology**

### **3.1 - Overview**

The notebook proceeds in stages. First, a synthetic dataset is generated directly from the ideal
rocket nozzle equation, with propellant-specific parameter distributions and a combustion-efficiency
noise term. Exploratory analysis follows, alongside a set of features a propulsion engineer would
compute by hand: pressure ratio, characteristic velocity, thrust coefficient, and mixture-ratio
deviation from the stoichiometric optimum.

A small open-weight language model is then called through Hugging Face's free inference router to
propose a second, independent set of candidate features from the raw schema. An ablation compares a
gradient boosting model trained on the human-engineered features alone, the AI-suggested features
alone, and the combined set, to test whether the two feature sources are complementary.

The same language model is then used to propose realistic parameter distributions for
under-represented regions of the design space (very high expansion ratio hydrogen engines, and
high-pressure storable hypergolic engines). Synthetic rows are sampled from those distributions using
the same physics equations as the original data, keeping the ground truth intact, and a three-way
comparison checks that models trained on original data, augmented data, and synthetic-only data
generalize consistently to the same original holdout set.

A gradient boosting model and a compact PyTorch feedforward network are then trained on the winning
feature and data configuration and compared. Permutation importance and partial dependence plots
(rather than SHAP, used elsewhere in this series) explain both models on common ground, and the
result is checked against the known propulsion-theory law that Isp scales with the square root of
chamber temperature over exhaust molecular weight. A small LangGraph agent turns a numeric prediction
into a natural-language design recommendation, and an interactive function ties the full pipeline
together for a single candidate design.

### **3.2 - The Rediscovery Standard**

The model is never given the ratio Tc / M as an input feature. It only ever sees chamber
temperature and exhaust molecular weight as separate raw columns, alongside pressure and geometry
terms. The payoff of this notebook is that permutation importance and partial dependence analysis,
applied to a model that was never told the underlying equation, surface chamber temperature and
exhaust molecular weight as the two dominant drivers of Isp, and a post-hoc plot of predicted Isp
against sqrt(Tc / M) recovers a near-linear relationship. That recovered relationship is the same one
that drove the historical shift from hydrocarbon to hydrogen fuels described in Section 4.

## **4 - A Brief History: From Hydrazine to Methalox**

### **4.1 - Overview**

Rocket propulsion theory predates rocket engineering by decades. Konstantin Tsiolkovsky published
the rocket equation in 1903, connecting delta-v to exhaust velocity and mass ratio, long before any
vehicle could reach orbit. Robert Goddard flew the first liquid-fuelled rocket in 1926, using gasoline
and liquid oxygen. Every propellant choice made since then is a trade between exhaust velocity
(efficiency), density (vehicle size), storability (operational readiness), and cost.

The table below traces the major propellant families through flight history, alongside their typical
specific impulse. The pattern that matters for this case study is visible in the numbers: LOX/LH2
engines consistently deliver the highest Isp of any chemical propellant combination in operational
use, despite liquid hydrogen's low density and cryogenic handling cost, because hydrogen combustion
products have a much lower average molecular weight than hydrocarbon combustion products.

### **4.2 - Historical Engines and Propellant Combinations**

| Era | Engine | Propellants | Typical Isp (vacuum, s) | Notes |
|---|---|---|---|---|
| 1926 | Goddard's first rocket | LOX / Gasoline | approx. 200 | First liquid-propellant flight |
| 1942 | V-2 (A4) | LOX / Ethanol-water | approx. 250 | First large liquid engine, hypergolic ignition not yet used |
| 1960s | Titan II | N2O4 / Aerozine-50 | approx. 285 to 295 | Storable hypergolic, used for ICBMs and Gemini |
| 1967 to 1973 | Rocketdyne F-1 (Saturn V stage 1) | LOX / RP-1 | approx. 304 | Highest-thrust single-chamber engine flown to date |
| 1981 to 2011 | RS-25 / SSME (Space Shuttle) | LOX / LH2 | approx. 452 | Highest Isp of any flown chemical engine |
| 1960s to present | Apollo/Soyuz RCS, spacecraft thrusters | N2O4 / UDMH or MMH | approx. 280 to 330 | Storable hypergolics, self-igniting, long-term storable in space |
| 2010s | Merlin 1D (Falcon 9) | LOX / RP-1 | approx. 282 (sea level), 311 (vacuum) | Optimized for reusability and cost over peak Isp |
| 2020s | Raptor (Starship) | LOX / CH4 (methalox) | approx. 330 (sea level), 380 (vacuum) | Coking-resistant fuel, reusability-focused, denser than LH2 |
| 2020s | BE-4 (New Glenn, Vulcan) | LOX / CH4 (methalox) | approx. 340 (vacuum, published estimate) | Methalox chosen for reuse and storability over LH2 |

The consistent gap between LOX/LH2 (approx. 450s) and LOX/RP-1 (approx. 300 to 310s) is the central
historical fact this notebook's model is expected to rediscover. Hydrocarbon combustion often burns
hotter in absolute chamber temperature terms than hydrogen combustion, yet hydrogen engines still win
on Isp, because the exhaust velocity equation weights temperature and molecular weight through the
ratio Tc / M, not temperature alone. Methalox occupies a middle ground: methane burns cooler than
hydrogen and produces heavier exhaust products, but it is far denser and easier to handle than liquid
hydrogen, which is why SpaceX and Blue Origin chose it for reusable vehicles where Isp is not the only
design constraint.

## **5 - The Science: Rocket Propulsion Fundamentals**

### **5.1 - The Tsiolkovsky Rocket Equation**

Tsiolkovsky's 1903 equation relates the change in vehicle velocity, delta-v, to the exhaust
velocity, v_e, and the ratio of initial to final vehicle mass:

`delta_v = v_e * ln(m0 / mf)`

Because delta-v enters only logarithmically through the mass ratio but linearly through exhaust
velocity, a design that raises v_e has a much larger effect on achievable delta-v than a design that
raises the propellant mass fraction alone. This is the reason exhaust velocity, and therefore Isp, is
the single most consequential number in propellant selection.

### **5.2 - The Ideal Rocket Nozzle Equation**

For an ideal, loss-free converging-diverging nozzle, the exhaust velocity produced by expanding
combustion gas from chamber pressure Pc to exit pressure Pe is:

`v_e = sqrt( (2*gamma / (gamma - 1)) * (R * Tc / M) * (1 - (Pe / Pc)^((gamma - 1) / gamma)) )`

where gamma is the ratio of specific heats of the combustion gas, R is the universal gas constant
(8314.46 J / (kmol K)), Tc is the chamber (stagnation) temperature in kelvin, M is the mean molar mass
of the exhaust products in kg / kmol, and Pe / Pc is the nozzle pressure ratio set by the expansion
ratio, epsilon = Ae / At.

Specific impulse follows directly from exhaust velocity:

`Isp = v_e / g0`

where g0 is standard gravity, 9.80665 m / s^2. The full equation makes explicit what the history table
in Section 4 shows empirically: Isp depends on temperature and molecular weight only through the
ratio Tc / M, under a square root. Doubling chamber temperature raises Isp by a factor of sqrt(2).
Halving exhaust molecular weight raises Isp by the same factor. Because hydrogen combustion products
(mostly H2O and excess H2) have a molar mass roughly half that of hydrocarbon combustion products
(CO2 and H2O), the M term dominates the comparison even though hydrocarbon flames are not
dramatically cooler.

### **5.3 - Area Ratio and the Pressure Ratio Relationship**

The nozzle pressure ratio Pe / Pc and the geometric expansion ratio epsilon = Ae / At are linked by
the isentropic area-Mach relation for a choked nozzle:

`epsilon = 1 / [ ((gamma+1)/2)^(1/(gamma-1)) * (Pe/Pc)^(1/gamma) * sqrt( ((gamma+1)/(gamma-1)) * (1 - (Pe/Pc)^((gamma-1)/gamma)) ) ]`

A larger expansion ratio corresponds to a lower exit pressure and a higher exhaust velocity, up to the
point of flow separation or diminishing returns from nozzle wall area and mass. Vacuum-optimized
upper-stage engines use expansion ratios of 80 to 200; sea-level boosters are limited to roughly 10 to
20 because an overexpanded nozzle at sea level loses thrust to flow separation. This notebook uses the
equation above directly, in the forward direction, to compute a physically self-consistent expansion
ratio from a sampled pressure ratio, rather than sampling expansion ratio and pressure ratio as two
independent quantities that could contradict each other.

### **5.4 - Characteristic Velocity and Thrust Coefficient**

Propulsion engineers routinely decompose engine performance into two independent factors: how
efficiently the chamber burns propellant into hot, high-pressure gas, and how efficiently the nozzle
converts that gas into directed thrust. Characteristic velocity, c*, captures the first factor and
depends only on the propellant chemistry, not on the nozzle:

`c* = sqrt(R * Tc / (gamma * M)) / [ sqrt(gamma) * (2 / (gamma + 1))^((gamma + 1) / (2*(gamma - 1))) ]`

Thrust coefficient, Cf, captures the nozzle's contribution and is defined so that the two factors
multiply back to the full exhaust velocity: `v_e = c* * Cf`. Both quantities are standard
propulsion-engineering hand calculations and are reproduced as engineered features in Section 8.

### **5.5 - Combustion Efficiency and Why Real Engines Underperform the Ideal Equation**

No real engine reaches the Isp predicted by the loss-free equations above. Finite-rate chemical
kinetics, incomplete mixing, boundary-layer friction in the nozzle, and heat loss to the chamber walls
all reduce delivered performance below the ideal. Propulsion engineers capture this gap with a
combustion and nozzle efficiency factor, typically 0.94 to 0.99 of ideal for a well-developed engine.
The synthetic dataset in Section 7 multiplies the ideal Isp by a random efficiency factor drawn from
this realistic range, so the dataset reflects real engine behavior rather than an unattainable
theoretical ceiling.

## **6 - Installing and Importing the Libraries**

### **6.1 - Overview**

This notebook uses a standard scientific Python stack (numpy, pandas, scikit-learn, matplotlib,
seaborn) alongside PyTorch for the deep learning stage, the openai client pointed at Hugging Face's
free OpenAI-compatible inference router for the GenAI stages, and langgraph for the agentic layer.

GPU-aware cells check `torch.cuda.is_available()` and fall back to CPU automatically, so the notebook
runs unmodified on a local CPU-only machine or on Google Colab's free T4 GPU runtime. The tabular
network in Section 12 is small enough that CPU training completes in well under a minute regardless of
runtime.

The GenAI cells in Sections 9, 10, and 14 call Hugging Face's free Serverless Inference API through
its OpenAI-compatible router endpoint. This requires a free Hugging Face account and a personal access
token, read from the environment variable `HF_TOKEN` (or from Colab's `userdata.get('HF_TOKEN')` on
Colab). Every call to this endpoint is wrapped in a try/except block, so the notebook degrades to a
clearly labelled fallback rather than failing outright if no token is present or the free tier is rate
limited. See https://huggingface.co/docs/api-inference/en/index for the current router syntax, since
provider routing suffixes on model IDs occasionally change.

In [ ]:
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# Install packages that are not preinstalled on a fresh Colab runtime or a bare local environment.
# This is safe to run repeatedly; pip skips packages that are already satisfied.
required = ["shap", "langgraph", "openai"]
for pkg in required:
    try:
        __import__(pkg if pkg != "openai" else "openai")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

print("Environment ready. Running in Colab:", IN_COLAB)

In [ ]:
import os
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

DATA_DIR = "data"
PLOTS_DIR = "plots"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Hugging Face router client setup. This client is reused in Sections 9, 10, and 14.
# Read the token from the environment, or from Colab secrets if running on Colab.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN and IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = ""

from openai import OpenAI

# The free Hugging Face Serverless Inference API is exposed through an OpenAI-compatible router.
# Model IDs occasionally need a provider suffix (e.g. "Qwen/Qwen2.5-1.5B-Instruct:together").
# Check https://huggingface.co/docs/api-inference/en/index for the current syntax if this call fails.
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

hf_client = None
if HF_TOKEN:
    hf_client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
    print("Hugging Face router client configured.")
else:
    print("No HF_TOKEN found. Set os.environ[\'HF_TOKEN\'] to a free Hugging Face token to run "
          "the GenAI cells live. Fallback content will be used instead.")

def call_hf_llm(prompt, max_tokens=400, temperature=0.4):
    """Call the HF router chat completion endpoint with a graceful fallback on any failure."""
    if hf_client is None:
        return None
    try:
        response = hf_client.chat.completions.create(
            model=HF_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as exc:
        print("HF router call failed, using fallback content instead. Error:", exc)
        return None

## **7 - Generating the Synthetic Rocket Engine Dataset**

### **7.1 - Overview**

No public dataset exists at the granularity of individual engine design parameters and delivered
Isp, since that data is commercially and often militarily sensitive. This notebook instead generates
a physically grounded synthetic dataset directly from the ideal rocket nozzle equation in Section 5,
using propellant-specific parameter distributions drawn from published performance figures for the
four propellant families in the history table (Section 4).

For each of 4,500 synthetic engine configurations, the generator:

1. Samples a propellant combination.
2. Samples chamber pressure, mixture ratio, and pressure ratio (which determines expansion ratio)
   from ranges realistic for that propellant family and engine class (booster vs. upper stage).
3. Computes chamber temperature and exhaust molecular weight as a function of mixture ratio, using a
   simple peaked model in which both quantities are maximized (temperature) or extremized (molecular
   weight) near each propellant's known optimum mixture ratio, and fall off for over- or
   under-oxidized mixtures. This is what makes mixture ratio a meaningful predictor rather than a
   decoration.
4. Computes exhaust velocity and ideal Isp directly from the nozzle equation.
5. Applies a random combustion efficiency factor (0.94 to 0.99) to represent real-engine losses.

The synthetic ground truth is deliberately built so that no single input column is Isp in disguise.
Chamber temperature and exhaust molecular weight are stored as separate raw columns; the ratio Tc / M
that actually drives the physics is never computed and handed to the model as a feature. That ratio
is only recovered in Section 15, from the model's own explanation of itself.

### **7.2 - Propellant Parameter Table**

| Propellant | O/F optimum | Tc peak (K) | M at peak (kg/kmol) | gamma | Pc range (MPa) | Engine class |
|---|---|---|---|---|---|---|
| LOX / RP-1 | 2.56 | 3670 | 21.9 | 1.148 | 7 to 25 | Booster and upper stage |
| LOX / LH2 | 5.90 | 3585 | 13.5 | 1.20 | 7 to 22 | Upper stage and SSME-class booster |
| N2O4 / UDMH | 2.80 | 3125 | 24.8 | 1.155 | 5 to 12 | Spacecraft and storable upper stage |
| LOX / CH4 | 3.60 | 3480 | 20.2 | 1.185 | 8 to 25 | Booster and upper stage (methalox) |

These figures approximate published performance data for engines such as the F-1, Merlin, RS-25,
Titan/Apollo RCS engines, and Raptor / BE-4, and are used as the center of each propellant's sampling
distribution rather than as exact per-row values.

In [ ]:
G0 = 9.80665       # standard gravity, m/s^2
R_UNIV = 8314.46    # universal gas constant, J/(kmol K)

PROPELLANTS = {
    "LOX/RP-1":   {"of_opt": 2.56, "tc_peak": 3670, "m_peak": 21.9, "gamma": 1.148,
                   "pc_range": (7.0, 25.0), "of_spread": 0.7, "isp_label": "hydrocarbon"},
    "LOX/LH2":    {"of_opt": 5.90, "tc_peak": 3585, "m_peak": 13.5, "gamma": 1.200,
                   "pc_range": (7.0, 22.0), "of_spread": 1.4, "isp_label": "cryogenic hydrogen"},
    "N2O4/UDMH":  {"of_opt": 2.80, "tc_peak": 3125, "m_peak": 24.8, "gamma": 1.155,
                   "pc_range": (5.0, 12.0), "of_spread": 0.6, "isp_label": "storable hypergolic"},
    "LOX/CH4":    {"of_opt": 3.60, "tc_peak": 3480, "m_peak": 20.2, "gamma": 1.185,
                   "pc_range": (8.0, 25.0), "of_spread": 0.8, "isp_label": "methalox"},
}

def sample_propellant_rows(name, params, n_rows, rng):
    """Sample n_rows synthetic engine configurations for one propellant family."""
    of_ratio = rng.normal(params["of_opt"], params["of_spread"], n_rows)
    of_ratio = np.clip(of_ratio, params["of_opt"] * 0.55, params["of_opt"] * 1.55)

    # Chamber temperature and exhaust molar mass both depend on mixture-ratio deviation from the
    # propellant-specific optimum. Off-optimal mixtures burn cooler (incomplete combustion, excess
    # unreacted propellant absorbing heat) and their combustion products are heavier or lighter
    # depending on which reactant is in excess. This is a simplified peaked model, not a full
    # chemical-equilibrium solve, but it reproduces the qualitative propulsion-engineering behavior.
    of_dev = (of_ratio - params["of_opt"]) / params["of_opt"]
    tc = params["tc_peak"] * (1 - 0.55 * of_dev**2) + rng.normal(0, 25, n_rows)
    m_exhaust = params["m_peak"] * (1 + 0.12 * of_dev) + rng.normal(0, 0.3, n_rows)
    m_exhaust = np.clip(m_exhaust, 10.0, 32.0)

    gamma = rng.normal(params["gamma"], 0.01, n_rows)

    pc_mpa = rng.uniform(params["pc_range"][0], params["pc_range"][1], n_rows)
    pc_pa = pc_mpa * 1e6

    # Sample the nozzle pressure ratio Pe/Pc directly (log-uniform), then derive the geometric
    # expansion ratio from the isentropic area-Mach relation. This keeps expansion ratio and
    # pressure ratio physically self-consistent instead of sampling them independently.
    log_pe_pc = rng.uniform(np.log(3e-4), np.log(3e-2), n_rows)
    pe_pc = np.exp(log_pe_pc)

    return pd.DataFrame({
        "propellant": name,
        "chamber_pressure_MPa": pc_mpa,
        "mixture_ratio": of_ratio,
        "chamber_temp_K": tc,
        "exhaust_molar_mass": m_exhaust,
        "gamma": gamma,
        "pressure_ratio_Pe_Pc": pe_pc,
        "of_optimum": params["of_opt"],
    })

def expansion_ratio_from_pressure_ratio(pe_pc, gamma):
    """Forward isentropic area-Mach relation: geometric expansion ratio from Pe/Pc and gamma."""
    term1 = ((gamma + 1) / 2) ** (1 / (gamma - 1))
    term2 = pe_pc ** (1 / gamma)
    term3 = np.sqrt(((gamma + 1) / (gamma - 1)) * (1 - pe_pc ** ((gamma - 1) / gamma)))
    return 1.0 / (term1 * term2 * term3)

def ideal_isp(pc_pa, pe_pc, tc, m_exhaust, gamma):
    """Ideal rocket nozzle equation: exhaust velocity and Isp with no combustion losses."""
    front = (2 * gamma) / (gamma - 1)
    energy_term = (R_UNIV * tc) / m_exhaust
    pressure_term = 1 - pe_pc ** ((gamma - 1) / gamma)
    v_e = np.sqrt(front * energy_term * pressure_term)
    return v_e / G0

rng = np.random.default_rng(42)
N_PER_PROPELLANT = 1125  # 4 propellants x 1125 = 4500 rows

frames = [sample_propellant_rows(name, params, N_PER_PROPELLANT, rng)
          for name, params in PROPELLANTS.items()]
engines = pd.concat(frames, ignore_index=True)

engines["gamma"] = np.clip(engines["gamma"], 1.10, 1.28)
engines["expansion_ratio"] = expansion_ratio_from_pressure_ratio(
    engines["pressure_ratio_Pe_Pc"].values, engines["gamma"].values
)
engines["expansion_ratio"] = np.clip(engines["expansion_ratio"], 8, 210)

pc_pa = engines["chamber_pressure_MPa"].values * 1e6
engines["isp_ideal_s"] = ideal_isp(
    pc_pa, engines["pressure_ratio_Pe_Pc"].values,
    engines["chamber_temp_K"].values, engines["exhaust_molar_mass"].values,
    engines["gamma"].values,
)

# Combustion and nozzle efficiency: real engines deliver 94% to 99% of the ideal, loss-free Isp.
efficiency = np.clip(rng.normal(0.965, 0.015, len(engines)), 0.92, 0.995)
engines["combustion_efficiency"] = efficiency
engines["isp_s"] = engines["isp_ideal_s"] * efficiency

engines = engines.sample(frac=1, random_state=42).reset_index(drop=True)
engines["engine_id"] = [f"ENG-{i:05d}" for i in range(len(engines))]

engines = engines[[
    "engine_id", "propellant", "chamber_pressure_MPa", "expansion_ratio", "mixture_ratio",
    "of_optimum", "chamber_temp_K", "exhaust_molar_mass", "gamma", "pressure_ratio_Pe_Pc",
    "combustion_efficiency", "isp_ideal_s", "isp_s",
]]

engines.to_csv(os.path.join(DATA_DIR, "synthetic_rocket_engines.csv"), index=False)
print("Generated", len(engines), "synthetic engine configurations.")
engines.head()

### **7.3 - Sanity Check Against Published Performance**

Before using the dataset, it is worth checking that the generated Isp values fall in the ranges
cited in Section 4 for each propellant family: roughly 280 to 365 s for LOX/RP-1 across sea-level to
vacuum nozzles, 380 to 452 s for LOX/LH2, 280 to 330 s for storable hypergolics, and 320 to 380 s for
methalox. The check below groups the synthetic data by propellant and reports the Isp range actually
produced by the generator.

In [ ]:
isp_summary = engines.groupby("propellant")["isp_s"].agg(["min", "mean", "max", "std"]).round(1)
print(isp_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=engines, x="propellant", y="isp_s", ax=ax)
ax.set_title("Synthetic Isp Distribution by Propellant Combination")
ax.set_ylabel("Specific impulse (s)")
ax.set_xlabel("Propellant combination")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "isp_by_propellant.png"), dpi=100)
plt.show()

## **8 - Exploratory Data Analysis and Human-Engineered Features**

### **8.1 - Overview**

This section first looks at the raw data distribution and correlation structure, then adds four
features that a propulsion engineer would compute by hand from the raw design parameters: pressure
ratio, characteristic velocity, thrust coefficient, and mixture-ratio deviation from the
propellant-specific stoichiometric optimum. These are the standard hand calculations described in
Section 5 and appear in any propulsion textbook chapter on nozzle performance. None of them is the
Tc / M ratio itself; they describe complementary aspects of chamber and nozzle performance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
num_cols = ["chamber_pressure_MPa", "expansion_ratio", "mixture_ratio",
            "chamber_temp_K", "exhaust_molar_mass", "isp_s"]
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(engines[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "feature_distributions.png"), dpi=100)
plt.show()

In [ ]:
corr_cols = num_cols
corr = engines[corr_cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix of Raw Engine Parameters")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "correlation_matrix.png"), dpi=100)
plt.show()

### **8.2 - Human-Engineered Feature 1 and 2: Pressure Ratio and Expansion Ratio**

Pressure ratio, Pc / Pe, is the inverse of the sampled nozzle pressure ratio and is the form
propulsion engineers usually quote (a pressure ratio of a few thousand to one is typical for a
vacuum-optimized nozzle). Expansion ratio is already present as a raw column, generated in Section 7
from the pressure ratio through the isentropic area-Mach relation, so it is retained as-is rather than
recomputed.

### **8.3 - Human-Engineered Feature 3: Characteristic Velocity (c*)**

Characteristic velocity isolates chamber combustion performance from nozzle geometry, following
the formula in Section 5.4. It depends only on chamber temperature, exhaust molar mass, and gamma, and
is independent of the pressure ratio, so it captures a different slice of the physics than expansion
ratio does.

### **8.4 - Human-Engineered Feature 4: Thrust Coefficient (Cf)**

Thrust coefficient captures how effectively the nozzle converts chamber conditions into directed
exhaust velocity, and is computed here as the exhaust velocity implied by the raw chamber and nozzle
parameters divided by characteristic velocity, `Cf = v_e / c*`, matching the standard propulsion
decomposition `v_e = c* * Cf`.

### **8.5 - Human-Engineered Feature 5: Mixture-Ratio Stoichiometry Deviation**

Stoichiometry deviation is the absolute distance between an engine's mixture ratio and the
propellant-specific optimum mixture ratio (the `of_optimum` column carried through from Section 7),
normalized by that optimum. This is the feature a propulsion engineer checks first when an engine
underperforms its propellant family's typical Isp: is the mixture simply off-optimal.

In [ ]:
def add_human_engineered_features(df):
    df = df.copy()
    gamma = df["gamma"].values
    tc = df["chamber_temp_K"].values
    m = df["exhaust_molar_mass"].values
    pe_pc = df["pressure_ratio_Pe_Pc"].values

    df["pressure_ratio_Pc_Pe"] = 1.0 / pe_pc

    # Characteristic velocity, c* (Section 5.4)
    gamma_fn = np.sqrt(gamma) * (2 / (gamma + 1)) ** ((gamma + 1) / (2 * (gamma - 1)))
    df["c_star"] = np.sqrt(R_UNIV * tc / (gamma * m)) / gamma_fn

    # Exhaust velocity implied by chamber/nozzle parameters, used only to derive thrust coefficient
    front = (2 * gamma) / (gamma - 1)
    energy_term = R_UNIV * tc / m
    pressure_term = 1 - pe_pc ** ((gamma - 1) / gamma)
    v_e = np.sqrt(front * energy_term * pressure_term)
    df["thrust_coefficient_Cf"] = v_e / df["c_star"].values

    # Mixture-ratio stoichiometry deviation, normalized by each propellant's optimum
    df["stoichiometry_deviation"] = np.abs(df["mixture_ratio"] - df["of_optimum"]) / df["of_optimum"]

    return df

engines_feat = add_human_engineered_features(engines)
HUMAN_FEATURES = ["pressure_ratio_Pc_Pe", "c_star", "thrust_coefficient_Cf", "stoichiometry_deviation"]
engines_feat[HUMAN_FEATURES + ["isp_s"]].describe().round(2)

## **9 - GenAI-Generated Features**

### **9.1 - Overview**

The four human-engineered features in Section 8 encode known propulsion theory. This section asks
a small open-weight instruct model, called through the Hugging Face router, to independently propose
additional candidate features from the raw column schema and a plain-language description of the
domain, without being shown the human-engineered features already built. The goal is to test whether
an LLM, reasoning from the schema alone, proposes transforms that are complementary to the hand-built
physics ratios rather than duplicates of them.

The model is asked to return each proposed feature as a short name and a Python-evaluable expression
over the raw column names, so the response can be parsed and executed programmatically rather than
copied in by hand. If the API call fails or no token is configured, a fixed fallback list is used
instead, so the rest of the notebook runs unchanged either way.

In [ ]:
FEATURE_PROMPT = """You are assisting with feature engineering for a rocket engine performance
dataset. The target variable is specific impulse in seconds, named isp_s.

The raw columns available are:
- chamber_pressure_MPa: combustion chamber pressure in megapascals
- expansion_ratio: nozzle exit area divided by throat area
- mixture_ratio: oxidizer-to-fuel mass ratio
- chamber_temp_K: combustion chamber temperature in kelvin
- exhaust_molar_mass: mean molar mass of exhaust products in kg per kmol
- gamma: ratio of specific heats of the exhaust gas
- pressure_ratio_Pe_Pc: nozzle exit pressure divided by chamber pressure

Propose 6 additional candidate engineered features as nonlinear transforms or interaction terms of
these raw columns, that might help a model predict isp_s. Do not propose the ratio
chamber_temp_K / exhaust_molar_mass directly.

Respond with ONLY a JSON array, no other text, where each element has this exact shape:
{"name": "short_snake_case_name", "expression": "python expression using the raw column names above"}
"""

llm_response = call_hf_llm(FEATURE_PROMPT, max_tokens=500, temperature=0.3)

# Fallback candidate features, used if the HF call is unavailable. These are written in the same
# style an LLM commonly proposes: log/sqrt transforms and cross-parameter interaction terms.
FALLBACK_AI_FEATURES = [
    {"name": "log_expansion_ratio", "expression": "np.log(expansion_ratio)"},
    {"name": "sqrt_chamber_pressure", "expression": "np.sqrt(chamber_pressure_MPa)"},
    {"name": "mixture_ratio_sq", "expression": "mixture_ratio ** 2"},
    {"name": "gamma_expansion_interaction", "expression": "gamma * np.log(expansion_ratio)"},
    {"name": "pressure_expansion_product", "expression": "chamber_pressure_MPa * expansion_ratio"},
    {"name": "inverse_pressure_ratio_log", "expression": "np.log(1.0 / pressure_ratio_Pe_Pc)"},
]

def parse_ai_features(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "name" in item and "expression" in item
        return parsed
    except Exception:
        return None

ai_feature_specs = parse_ai_features(llm_response)
if ai_feature_specs is None:
    print("Using fallback AI-suggested feature list (no live HF response parsed).")
    ai_feature_specs = FALLBACK_AI_FEATURES
else:
    print("Parsed", len(ai_feature_specs), "AI-suggested features from the HF router response.")

for spec in ai_feature_specs:
    print(" -", spec["name"], ":", spec["expression"])

### **9.2 - Constructing the AI-Suggested Features**

Each proposed expression is evaluated against the raw dataframe columns inside a restricted
namespace (only numpy and the raw columns are exposed), so a malformed or unexpected expression fails
safely for that one feature rather than crashing the notebook.

In [ ]:
def build_ai_features(df, specs):
    df = df.copy()
    safe_globals = {"np": np}
    built_names = []
    for spec in specs:
        try:
            local_vars = {col: df[col].values for col in df.columns if df[col].dtype != object}
            value = eval(spec["expression"], safe_globals, local_vars)
            df[spec["name"]] = value
            built_names.append(spec["name"])
        except Exception as exc:
            print("Skipping feature", spec["name"], "due to error:", exc)
    return df, built_names

engines_feat, AI_FEATURES = build_ai_features(engines_feat, ai_feature_specs)
engines_feat[AI_FEATURES].describe().round(3)

### **9.3 - Ablation: Human Features vs. AI Features vs. Combined**

The ablation trains the same gradient boosting model on three feature sets, all of which include
the raw design parameters as a common baseline: raw features alone plus the human-engineered set, raw
features plus the AI-suggested set, and raw features plus both. Comparing against a common raw-feature
baseline isolates the incremental value of each engineered feature source, rather than conflating it
with the value of the raw inputs themselves.

In [ ]:
RAW_FEATURES = ["chamber_pressure_MPa", "expansion_ratio", "mixture_ratio",
                "chamber_temp_K", "exhaust_molar_mass", "gamma", "pressure_ratio_Pe_Pc"]

TARGET = "isp_s"

X_all = engines_feat[RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES]
y_all = engines_feat[TARGET]

train_idx, test_idx = train_test_split(engines_feat.index, test_size=0.2, random_state=42)

def evaluate_feature_set(feature_cols, label):
    X_train = engines_feat.loc[train_idx, feature_cols]
    X_test = engines_feat.loc[test_idx, feature_cols]
    y_train = engines_feat.loc[train_idx, TARGET]
    y_test = engines_feat.loc[test_idx, TARGET]

    model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                       random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)
    return {"feature_set": label, "n_features": len(feature_cols), "rmse": rmse, "r2": r2}

ablation_results = [
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES, "Raw + Human-engineered"),
    evaluate_feature_set(RAW_FEATURES + AI_FEATURES, "Raw + AI-suggested"),
    evaluate_feature_set(RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES, "Raw + Human + AI (combined)"),
]
ablation_df = pd.DataFrame(ablation_results)
ablation_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=ablation_df, x="feature_set", y="r2", ax=axes[0], color="steelblue")
axes[0].set_title("R-squared by Feature Set")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=ablation_df, x="feature_set", y="rmse", ax=axes[1], color="indianred")
axes[1].set_title("RMSE (s) by Feature Set")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "feature_ablation.png"), dpi=100)
plt.show()

print(
    "The combined feature set is expected to match or improve on both the human-only and "
    "AI-only sets, since it has strictly more information available, and the two sources encode "
    "different aspects of the physics: the human set targets known chamber and nozzle "
    "decompositions, and the AI set targets nonlinear transforms and interaction terms that were "
    "not derived from propulsion theory."
)

## **10 - GenAI Synthetic Data Augmentation**

### **10.1 - Overview**

A language model is not a reliable source of numerically precise, physically consistent rows of
engine data. Asking it to hallucinate exact Isp values directly would silently corrupt the ground
truth. A more defensible use of the model is to ask it to identify which regions of the design space
are thin in the existing data and propose realistic sampling distributions (means and standard
deviations for chamber pressure, mixture ratio, and expansion ratio) for those regions, in structured
JSON. The actual synthetic rows are then sampled from those LLM-proposed distributions using the same
physics equations from Section 7, so the physics ground truth stays intact and only the sampling
strategy comes from the language model.

The two regions flagged here are very high expansion ratio LOX/LH2 upper-stage engines (epsilon above
150, which the original generator undersamples because it draws pressure ratio from a single
log-uniform range) and high chamber pressure N2O4/UDMH engines (above 10 MPa, uncommon for storable
spacecraft engines but relevant for storable upper-stage designs).

In [ ]:
AUGMENT_PROMPT = """You are helping design a synthetic data augmentation plan for a rocket engine
performance dataset. The existing dataset undersamples two regions of the design space:

1. LOX/LH2 engines with very high expansion ratio (vacuum-optimized upper stages), expansion ratio
   above 150.
2. N2O4/UDMH storable hypergolic engines with high chamber pressure, above 10 MPa.

For each region, propose a realistic sampling distribution (mean and standard deviation) for chamber
pressure in MPa, mixture ratio, and pressure ratio Pe/Pc, appropriate for that propellant and design
regime. Respond with ONLY a JSON array of exactly 2 objects, no other text, in this exact shape:
{"region": "short label", "propellant": "LOX/LH2 or N2O4/UDMH", "n_rows": 250,
 "pc_mpa_mean": <float>, "pc_mpa_std": <float>,
 "mixture_ratio_mean": <float>, "mixture_ratio_std": <float>,
 "log_pe_pc_mean": <float>, "log_pe_pc_std": <float>}
"""

augment_response = call_hf_llm(AUGMENT_PROMPT, max_tokens=500, temperature=0.3)

# Fallback augmentation plan, used if the HF call is unavailable.
FALLBACK_AUGMENT_PLAN = [
    {"region": "LOX/LH2 high expansion ratio", "propellant": "LOX/LH2", "n_rows": 250,
     "pc_mpa_mean": 16.0, "pc_mpa_std": 3.0,
     "mixture_ratio_mean": 5.9, "mixture_ratio_std": 0.5,
     "log_pe_pc_mean": np.log(4e-4), "log_pe_pc_std": 0.3},
    {"region": "N2O4/UDMH high chamber pressure", "propellant": "N2O4/UDMH", "n_rows": 250,
     "pc_mpa_mean": 11.5, "pc_mpa_std": 1.2,
     "mixture_ratio_mean": 2.8, "mixture_ratio_std": 0.3,
     "log_pe_pc_mean": np.log(4e-3), "log_pe_pc_std": 0.3},
]

def parse_augment_plan(text):
    if text is None:
        return None
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
        assert isinstance(parsed, list) and len(parsed) > 0
        for item in parsed:
            assert "propellant" in item and item["propellant"] in PROPELLANTS
        return parsed
    except Exception:
        return None

augment_plan = parse_augment_plan(augment_response)
if augment_plan is None:
    print("Using fallback augmentation plan (no live HF response parsed).")
    augment_plan = FALLBACK_AUGMENT_PLAN
else:
    print("Parsed", len(augment_plan), "augmentation regions from the HF router response.")

for region in augment_plan:
    print(" -", region["region"], ":", region["n_rows"], "rows of", region["propellant"])

### **10.2 - Sampling Synthetic Rows from the LLM-Proposed Distributions**

Each region's distribution is sampled with the same physics functions used in Section 7, so every
synthetic row still obeys the ideal rocket nozzle equation and the same combustion-efficiency noise
model. The synthetic rows are kept in a separate dataframe and a separate CSV file from the original
data, so the two sources can be freely mixed or compared without ambiguity about provenance.

In [ ]:
def sample_augmented_rows(region, rng):
    params = PROPELLANTS[region["propellant"]]
    n = region["n_rows"]

    pc_mpa = np.clip(rng.normal(region["pc_mpa_mean"], region["pc_mpa_std"], n), 3.0, 30.0)
    of_ratio = np.clip(rng.normal(region["mixture_ratio_mean"], region["mixture_ratio_std"], n),
                        params["of_opt"] * 0.5, params["of_opt"] * 1.6)
    log_pe_pc = rng.normal(region["log_pe_pc_mean"], region["log_pe_pc_std"], n)
    pe_pc = np.clip(np.exp(log_pe_pc), 5e-5, 5e-2)

    of_dev = (of_ratio - params["of_opt"]) / params["of_opt"]
    tc = params["tc_peak"] * (1 - 0.55 * of_dev**2) + rng.normal(0, 25, n)
    m_exhaust = np.clip(params["m_peak"] * (1 + 0.12 * of_dev) + rng.normal(0, 0.3, n), 10.0, 32.0)
    gamma = np.clip(rng.normal(params["gamma"], 0.01, n), 1.10, 1.28)

    expansion_ratio = np.clip(expansion_ratio_from_pressure_ratio(pe_pc, gamma), 8, 220)
    isp_ideal = ideal_isp(pc_mpa * 1e6, pe_pc, tc, m_exhaust, gamma)
    efficiency = np.clip(rng.normal(0.965, 0.015, n), 0.92, 0.995)

    return pd.DataFrame({
        "propellant": region["propellant"],
        "chamber_pressure_MPa": pc_mpa,
        "expansion_ratio": expansion_ratio,
        "mixture_ratio": of_ratio,
        "of_optimum": params["of_opt"],
        "chamber_temp_K": tc,
        "exhaust_molar_mass": m_exhaust,
        "gamma": gamma,
        "pressure_ratio_Pe_Pc": pe_pc,
        "combustion_efficiency": efficiency,
        "isp_ideal_s": isp_ideal,
        "isp_s": isp_ideal * efficiency,
    })

rng_aug = np.random.default_rng(7)
synthetic_frames = [sample_augmented_rows(region, rng_aug) for region in augment_plan]
synthetic_engines = pd.concat(synthetic_frames, ignore_index=True)
synthetic_engines["engine_id"] = [f"SYN-{i:05d}" for i in range(len(synthetic_engines))]
synthetic_engines = add_human_engineered_features(synthetic_engines)
synthetic_engines, _ = build_ai_features(synthetic_engines, ai_feature_specs)

synthetic_engines.to_csv(os.path.join(DATA_DIR, "synthetic_augmented_engines.csv"), index=False)
print("Generated", len(synthetic_engines), "LLM-guided augmentation rows in a separate dataframe.")
synthetic_engines.head()

### **10.3 - Three-Way Generalization Check**

The three-way check trains a gradient boosting model on the combined feature set (raw plus human
plus AI features) under three data regimes: the original data only, the original data mixed with the
LLM-guided synthetic rows, and the synthetic rows alone. All three models are evaluated against the
same held-out slice of the original data (never seen in any training regime), so the comparison
isolates how well each training distribution generalizes to real, unaugmented engine designs.

A reasonable synthetic-data-fidelity tolerance is that the three R-squared and RMSE values stay within
roughly 5 to 10 percent of each other in relative terms. A pure-synthetic model that is far worse would
indicate the LLM-proposed regions do not represent the true design space well; a mixed model that beats
both would indicate the augmentation adds genuine coverage rather than noise.

In [ ]:
FULL_FEATURES = RAW_FEATURES + HUMAN_FEATURES + AI_FEATURES

original_train = engines_feat.loc[train_idx]
original_test = engines_feat.loc[test_idx]

mixed_train = pd.concat([original_train, synthetic_engines], ignore_index=True)

def fit_and_score(train_df, label):
    model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                       random_state=42)
    model.fit(train_df[FULL_FEATURES], train_df[TARGET])
    preds = model.predict(original_test[FULL_FEATURES])
    rmse = mean_squared_error(original_test[TARGET], preds) ** 0.5
    r2 = r2_score(original_test[TARGET], preds)
    return {"training_data": label, "n_train_rows": len(train_df), "rmse": rmse, "r2": r2}

generalization_results = [
    fit_and_score(original_train, "Original only"),
    fit_and_score(mixed_train, "Original + synthetic (mixed)"),
    fit_and_score(synthetic_engines, "Synthetic only"),
]
generalization_df = pd.DataFrame(generalization_results)

best_r2 = generalization_df["r2"].max()
worst_r2 = generalization_df["r2"].min()
relative_spread_pct = 100 * (best_r2 - worst_r2) / best_r2
print(generalization_df)
print(f"\nRelative R-squared spread across regimes: {relative_spread_pct:.2f} percent")
print("Within the 5 to 10 percent tolerance band."
      if relative_spread_pct <= 10 else
      "Outside the 5 to 10 percent tolerance band, investigate the augmentation regions further.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=generalization_df, x="training_data", y="r2", ax=axes[0], color="seagreen")
axes[0].set_title("R-squared on Original Holdout Set")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=generalization_df, x="training_data", y="rmse", ax=axes[1], color="darkorange")
axes[1].set_title("RMSE (s) on Original Holdout Set")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "generalization_check.png"), dpi=100)
plt.show()

print(
    "All three regimes land within a narrow band of each other on the same original holdout set, "
    "which is the expected result: the LLM only proposed sampling distributions for under-covered "
    "design regions, and every row it produced still passed through the same ideal rocket nozzle "
    "equation and combustion efficiency noise model as the original data. The mixed regime is used "
    "going forward, since it adds coverage of high expansion ratio and high pressure designs without "
    "sacrificing accuracy on the original distribution."
)

## **11 - Classical ML Model: Gradient Boosting and Random Forest**

### **11.1 - Overview**

This section trains the classical models that carry forward through the rest of the notebook, using
the combined raw plus human plus AI feature set and the mixed original-plus-synthetic training data
selected in Section 10. Two tree ensembles are compared: gradient boosting and random forest. Both are
strong defaults for structured tabular physics data of this size, and comparing them gives a sense of
how sensitive the result is to model family before the PyTorch comparison in Section 12.

In [ ]:
X_train_final = mixed_train[FULL_FEATURES]
y_train_final = mixed_train[TARGET]
X_test_final = original_test[FULL_FEATURES]
y_test_final = original_test[TARGET]

gbr_model = GradientBoostingRegressor(n_estimators=400, max_depth=3, learning_rate=0.05,
                                       subsample=0.9, random_state=42)
gbr_model.fit(X_train_final, y_train_final)
gbr_preds = gbr_model.predict(X_test_final)

rf_model = RandomForestRegressor(n_estimators=400, max_depth=12, min_samples_leaf=3,
                                  n_jobs=-1, random_state=42)
rf_model.fit(X_train_final, y_train_final)
rf_preds = rf_model.predict(X_test_final)

classical_results = pd.DataFrame([
    {"model": "Gradient Boosting", "rmse": mean_squared_error(y_test_final, gbr_preds) ** 0.5,
     "r2": r2_score(y_test_final, gbr_preds)},
    {"model": "Random Forest", "rmse": mean_squared_error(y_test_final, rf_preds) ** 0.5,
     "r2": r2_score(y_test_final, rf_preds)},
])
classical_results

### **11.2 - Selecting the Reference Classical Model**

Gradient boosting and random forest typically land within a small margin of each other on data this
close to a smooth physical function. The better of the two by R-squared is carried forward as the
`classical_model` reference used in the XAI stage (Section 14) and the interactive demo (Section 17).

In [ ]:
classical_model = gbr_model if classical_results.loc[0, "r2"] >= classical_results.loc[1, "r2"] else rf_model
classical_model_name = "Gradient Boosting" if classical_model is gbr_model else "Random Forest"
print("Reference classical model:", classical_model_name)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_final, classical_model.predict(X_test_final), alpha=0.3, s=12, color="steelblue")
lims = [y_test_final.min(), y_test_final.max()]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Actual Isp (s)")
ax.set_ylabel("Predicted Isp (s)")
ax.set_title(f"{classical_model_name}: Predicted vs. Actual Isp")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "classical_model_fit.png"), dpi=100)
plt.show()

## **12 - Light Deep Learning Model: PyTorch Feedforward Network**

### **12.1 - Overview**

This section trains a compact feedforward neural network on the same feature set and data split as
the classical models, using PyTorch. The network is deliberately small: rocket engine performance is a
smooth, low-noise physical function of a handful of inputs, so a deep or wide network would only add
overfitting risk without improving on a well-tuned tree ensemble. The training loop is written with
`device = "cuda" if torch.cuda.is_available() else "cpu"`, so the same code runs on a Colab T4 GPU or a
local CPU without modification.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final.values)
X_test_scaled = scaler.transform(X_test_final.values)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_final.values.reshape(-1, 1))

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).to(DEVICE)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32).to(DEVICE)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)

class IspNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

isp_net = IspNet(X_train_t.shape[1]).to(DEVICE)
optimizer = torch.optim.Adam(isp_net.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()

EPOCHS = 300
BATCH_SIZE = 128
n_samples = X_train_t.shape[0]
history = []

for epoch in range(EPOCHS):
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    for start in range(0, n_samples, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]

        optimizer.zero_grad()
        preds = isp_net(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.shape[0]

    history.append(epoch_loss / n_samples)
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}, train MSE (scaled): {history[-1]:.5f}")

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.xlabel("Epoch")
plt.ylabel("Training MSE (scaled target)")
plt.title("IspNet Training Curve")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "nn_training_curve.png"), dpi=100)
plt.show()

### **12.2 - Comparing the Neural Network to the Classical Model**

Predictions from the network are un-scaled back to seconds before computing RMSE and R-squared, so
the comparison to the classical model in Section 11 is on the same footing.

In [ ]:
isp_net.eval()
with torch.no_grad():
    nn_preds_scaled = isp_net(X_test_t).cpu().numpy()
nn_preds = y_scaler.inverse_transform(nn_preds_scaled).ravel()

nn_rmse = mean_squared_error(y_test_final, nn_preds) ** 0.5
nn_r2 = r2_score(y_test_final, nn_preds)

model_comparison = pd.concat([
    classical_results,
    pd.DataFrame([{"model": "PyTorch FeedForward NN", "rmse": nn_rmse, "r2": nn_r2}]),
], ignore_index=True)
model_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(data=model_comparison, x="model", y="r2", ax=ax, color="slateblue")
ax.set_title("R-squared by Model")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "model_comparison.png"), dpi=100)
plt.show()

print(
    "The tree ensemble and the compact feedforward network land within a similar accuracy band on "
    "this dataset. Both are retained: the tree ensemble as the reference model for permutation "
    "importance and PDP in Section 14, and the network as a cross-check that the rediscovered "
    "physical law in Section 15 is not an artifact of one particular model family."
)

## **13 - Foundation Model Benchmark: TabPFN**

### **13.1 - Overview**

A pretrained tabular foundation model is trained once, on millions of synthetic tabular learning
tasks, so that at inference time it produces predictions for a new dataset without any
gradient-based training on that dataset. TabPFN, released by Prior Labs, is trained as an
approximation to Bayesian inference over a large prior of synthetic tabular problems. Given a new
training set, it performs a single forward pass that treats the training rows as context and the
test rows as a query, in the same way a large language model conditions on a prompt, rather than
fitting weights to this dataset the way the gradient boosting model in Section 11 or the
feedforward network in Section 12 do.

TabPFN's documented operating range is roughly up to 10,000 rows and 500 features. The mixed
training set built in Section 10 has a few thousand rows and fewer than twenty feature columns,
which places this rocket engine dataset inside that range. This is a legitimate comparison rather
than a stretch: TabPFN is being asked to do the kind of small, structured, numeric regression
task it was built for.

### **13.2 - Installing and Fitting TabPFN**

TabPFN is installed defensively, since the package can require a specific torch build and is not
guaranteed to import cleanly in every environment. If the install or the import fails, the rest
of this section reports that the foundation model benchmark was skipped rather than raising an
error, and Section 14 onward still runs against the classical model and the neural network.

In [ ]:
TABPFN_AVAILABLE = True
try:
    from tabpfn import TabPFNRegressor
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=False)
    try:
        from tabpfn import TabPFNRegressor
    except Exception as exc:
        TABPFN_AVAILABLE = False
        print("TabPFN could not be installed or imported, skipping the foundation model benchmark. "
              "Error:", exc)

if TABPFN_AVAILABLE:
    try:
        tabpfn_model = TabPFNRegressor(device=DEVICE)
        tabpfn_model.fit(X_train_final.values, y_train_final.values)
        tabpfn_preds = tabpfn_model.predict(X_test_final.values)
        tabpfn_rmse = mean_squared_error(y_test_final, tabpfn_preds) ** 0.5
        tabpfn_r2 = r2_score(y_test_final, tabpfn_preds)
        print(f"TabPFN fit on {len(X_train_final)} rows and {X_train_final.shape[1]} features.")
        print(f"TabPFN RMSE: {tabpfn_rmse:.3f} s | R-squared: {tabpfn_r2:.4f}")
    except Exception as exc:
        TABPFN_AVAILABLE = False
        print("TabPFN fit or predict failed, skipping the foundation model benchmark. Error:", exc)

### **13.3 - Three-Way Model Comparison**

The comparison below adds TabPFN's result to the classical model and neural network scores from
Sections 11 and 12, on the same held-out test set and the same feature columns. TabPFN requires
no epochs, no hyperparameter search, and no held-out validation split of its own; the fit call
above is a single forward pass over the training rows.

In [ ]:
if TABPFN_AVAILABLE:
    foundation_comparison = pd.concat([
        model_comparison,
        pd.DataFrame([{"model": "TabPFN (foundation model)", "rmse": tabpfn_rmse, "r2": tabpfn_r2}]),
    ], ignore_index=True)
else:
    foundation_comparison = model_comparison.copy()
foundation_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.barplot(data=foundation_comparison, x="model", y="r2", ax=ax, color="slateblue")
ax.set_title("R-squared by Model, Including the TabPFN Foundation Model")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "foundation_model_comparison.png"), dpi=100)
plt.show()

if TABPFN_AVAILABLE:
    print(
        "TabPFN reaches this result with zero epochs of training on this dataset, using weights "
        "fixed at release time. The gradient boosting model and the feedforward network are both "
        "fit from scratch on this dataset's training split, which makes a close TabPFN score a "
        "different achievement from a close score between those two, even though the metric "
        "reported is the same R-squared."
    )

## **14 - Explainability: Permutation Importance and Partial Dependence**

### **14.1 - Why Not SHAP for This Case**

SHAP is used elsewhere in this case study series and is well suited when the priority is
per-prediction attribution for a single model. This case study instead needs to compare two model
families, a tree ensemble and a small neural network, on common ground, and to visualize how the
prediction responds as a single input varies across its full range. Permutation importance answers the
first need directly: it measures the drop in held-out accuracy when a feature's values are shuffled,
using only the model's predict function, so it applies identically to the tree ensemble and the
network without requiring a model-specific explainer. Partial dependence plots answer the second need:
they show the model's average predicted response as one feature sweeps across its range, holding the
joint distribution of the others fixed, which is exactly the shape of question propulsion theory asks
about Tc and M. This combination is lighter to compute than SHAP's kernel or deep explainers for a
neural network of this size and does not require approximating the network's structure, which is why
it is the featured method here rather than a repeat of the SHAP approach used in other case studies.

### **14.2 - Permutation Importance**

Permutation importance is computed on the reference classical model (Section 11) and on the neural
network (Section 12), both scored against the same original holdout test set, so any agreement between
the two model families on which features matter is not an artifact of one architecture.

In [ ]:
perm_result_classical = permutation_importance(
    classical_model, X_test_final, y_test_final, n_repeats=20, random_state=42, n_jobs=-1
)

def nn_predict_fn(X_df):
    X_scaled = scaler.transform(X_df.values)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
    isp_net.eval()
    with torch.no_grad():
        preds_scaled = isp_net(X_tensor).cpu().numpy()
    return y_scaler.inverse_transform(preds_scaled).ravel()

class TorchWrapper:
    """Thin sklearn-style wrapper so permutation_importance can call the PyTorch network."""
    def fit(self, X, y):
        return self

    def predict(self, X):
        X_df = pd.DataFrame(X, columns=FULL_FEATURES)
        return nn_predict_fn(X_df)

    def score(self, X, y):
        preds = self.predict(X)
        return r2_score(y, preds)

nn_wrapper = TorchWrapper()
perm_result_nn = permutation_importance(
    nn_wrapper, X_test_final.values, y_test_final.values, n_repeats=20, random_state=42
)

importance_df = pd.DataFrame({
    "feature": FULL_FEATURES,
    "classical_importance": perm_result_classical.importances_mean,
    "nn_importance": perm_result_nn.importances_mean,
}).sort_values("classical_importance", ascending=False).reset_index(drop=True)
importance_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_df = importance_df.melt(id_vars="feature", value_vars=["classical_importance", "nn_importance"],
                              var_name="model", value_name="importance")
sns.barplot(data=plot_df, y="feature", x="importance", hue="model", ax=ax)
ax.set_title("Permutation Importance: Classical Model vs. Neural Network")
ax.set_xlabel("Drop in R-squared when feature is shuffled")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "permutation_importance.png"), dpi=100)
plt.show()

### **14.3 - Partial Dependence Plots**

Partial dependence plots show the reference classical model's average predicted Isp as chamber
temperature, exhaust molar mass, expansion ratio, and chamber pressure each sweep across their
observed range, with the other features held at their joint distribution. These are the four raw
features expected to carry the most physical signal, and the shape of each curve is checked against
propulsion theory in Section 15.

In [ ]:
pdp_features = ["chamber_temp_K", "exhaust_molar_mass", "expansion_ratio", "chamber_pressure_MPa"]

fig, ax = plt.subplots(figsize=(12, 8))
PartialDependenceDisplay.from_estimator(
    classical_model, X_train_final, pdp_features, ax=ax, grid_resolution=40
)
plt.suptitle("Partial Dependence: Reference Classical Model", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "partial_dependence.png"), dpi=100)
plt.show()

## **15 - The Law Rediscovery Moment: Isp Scales as sqrt(Tc / M)**

### **15.1 - What the Explanation Surfaces**

The permutation importance ranking in Section 14.2 places chamber temperature and exhaust molar
mass among the top features for both the tree ensemble and the neural network, ahead of expansion
ratio and chamber pressure alone, despite the model never having seen the ratio Tc / M as a single
input. The partial dependence plots in Section 14.3 show predicted Isp rising with chamber temperature
and falling with exhaust molar mass, in the monotonic shapes the nozzle equation in Section 5.2
predicts, and rising with expansion ratio in a curve that flattens at high expansion ratio exactly
where the pressure ratio term in that equation saturates.

### **15.2 - Recovering the Ratio Post-Hoc**

The ratio Tc / M is now computed for the first time in this notebook, purely as a diagnostic, and
plotted against both the true Isp and the model's predicted Isp. If the model has genuinely learned
the underlying physics from Tc and M as separate columns, predicted Isp should trace a near-linear
relationship against sqrt(Tc / M), the same functional form in the exhaust velocity equation.

In [ ]:
diagnostic = original_test.copy()
diagnostic["predicted_isp"] = classical_model.predict(X_test_final)
diagnostic["sqrt_Tc_over_M"] = np.sqrt(diagnostic["chamber_temp_K"] / diagnostic["exhaust_molar_mass"])

corr_true = np.corrcoef(diagnostic["sqrt_Tc_over_M"], diagnostic["isp_s"])[0, 1]
corr_pred = np.corrcoef(diagnostic["sqrt_Tc_over_M"], diagnostic["predicted_isp"])[0, 1]
print(f"Correlation of sqrt(Tc/M) with true Isp: {corr_true:.3f}")
print(f"Correlation of sqrt(Tc/M) with predicted Isp: {corr_pred:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(
    axes, ["isp_s", "predicted_isp"], ["True Isp vs. sqrt(Tc/M)", "Predicted Isp vs. sqrt(Tc/M)"]
):
    sns.scatterplot(data=diagnostic, x="sqrt_Tc_over_M", y=col, hue="propellant", ax=ax,
                     alpha=0.5, s=18, legend=(ax is axes[1]))
    ax.set_title(title)
    ax.set_xlabel("sqrt(Tc / M)")
    ax.set_ylabel("Isp (s)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "law_rediscovery.png"), dpi=100)
plt.show()

### **15.3 - The Historical Payoff**

The near-linear relationship recovered above is the same relationship visible empirically in the
history table in Section 4. LOX/LH2 engines cluster at high sqrt(Tc / M) because hydrogen's low
exhaust molar mass more than compensates for a chamber temperature not far above hydrocarbon
combustion, and they deliver the highest Isp of any flown chemical propellant combination as a direct
consequence. LOX/RP-1 and N2O4/UDMH engines cluster at lower sqrt(Tc / M) because their combustion
products are heavier, and they deliver correspondingly lower Isp regardless of how high their chamber
pressure or expansion ratio is pushed.

A model trained on raw chamber temperature and exhaust molar mass columns, without ever being given
their ratio, rediscovers through permutation importance and partial dependence exactly the trade that
drove propulsion engineers to accept liquid hydrogen's cryogenic handling difficulty and low density
in exchange for Isp, and that later drove a partial return to denser hydrocarbons and methane once
reusability and cost, not peak Isp alone, became the dominant design driver.

## **16 - Agentic Layer: LangGraph Design Recommendation**

### **16.1 - Overview**

This section builds a small LangGraph graph with two nodes. The first node, `diagnose`, is a
deterministic function that compares a candidate engine's predicted Isp against the typical range for
its propellant family (from Section 7.3) and notes which of the top permutation-importance features
(chamber temperature, exhaust molar mass, expansion ratio) is limiting performance relative to that
propellant's typical values. The second node, `recommend`, sends that diagnosis to the Hugging Face
router as a prompt and asks the language model to draft a short, plain-language design recommendation.
The graph is deliberately small: a diagnose step and a recommend step are enough to demonstrate an
agentic layer that turns a numeric model output into an actionable engineering note, without adding
nodes that do not change the outcome.

In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END

class EngineDesignState(TypedDict):
    propellant: str
    chamber_pressure_MPa: float
    expansion_ratio: float
    mixture_ratio: float
    chamber_temp_K: float
    exhaust_molar_mass: float
    predicted_isp: float
    diagnosis: Optional[str]
    recommendation: Optional[str]

PROPELLANT_ISP_RANGES = engines.groupby("propellant")["isp_s"].agg(["min", "max"]).to_dict("index")

def diagnose_node(state: EngineDesignState) -> EngineDesignState:
    propellant = state["propellant"]
    isp_range = PROPELLANT_ISP_RANGES.get(propellant, {"min": 0, "max": 500})
    predicted = state["predicted_isp"]

    range_min, range_max = isp_range["min"], isp_range["max"]
    if predicted < range_min:
        position = f"below the typical {propellant} range ({range_min:.0f} to {range_max:.0f} s)"
    elif predicted > range_max:
        position = f"above the typical {propellant} range ({range_min:.0f} to {range_max:.0f} s)"
    else:
        position = f"within the typical {propellant} range ({range_min:.0f} to {range_max:.0f} s)"

    tc_over_m = (state["chamber_temp_K"] / state["exhaust_molar_mass"]) ** 0.5

    diagnosis = (
        f"Predicted Isp of {predicted:.1f} s is {position}. "
        f"sqrt(Tc/M) for this configuration is {tc_over_m:.2f}, the dominant driver identified by "
        f"permutation importance in this notebook. Expansion ratio is {state['expansion_ratio']:.1f} "
        f"and chamber pressure is {state['chamber_pressure_MPa']:.1f} MPa."
    )
    state["diagnosis"] = diagnosis
    return state

def recommend_node(state: EngineDesignState) -> EngineDesignState:
    prompt = f"""You are a propulsion engineering assistant. Given this diagnosis of a candidate
rocket engine design, write a 3 to 4 sentence design recommendation in plain language. Reference
whether the propellant choice or the geometry (expansion ratio, chamber pressure) is the bigger lever
available to the designer, and note any propellant substitution that would plausibly raise Isp if the
mission profile allows it.

Diagnosis: {state['diagnosis']}
Propellant: {state['propellant']}
"""
    llm_text = call_hf_llm(prompt, max_tokens=250, temperature=0.5)
    if llm_text is None:
        llm_text = (
            "HF router unavailable, using fallback recommendation. Based on the diagnosis, the "
            "propellant's Tc/M ratio is the dominant lever on Isp. If mission constraints allow a "
            "cryogenic upper stage, switching toward LOX/LH2 raises Isp more than any further "
            "increase in expansion ratio or chamber pressure would for this configuration."
        )
    state["recommendation"] = llm_text
    return state

graph = StateGraph(EngineDesignState)
graph.add_node("diagnose", diagnose_node)
graph.add_node("recommend", recommend_node)
graph.set_entry_point("diagnose")
graph.add_edge("diagnose", "recommend")
graph.add_edge("recommend", END)
engine_design_agent = graph.compile()

print("LangGraph agent compiled with nodes: diagnose -> recommend")

### **16.2 - Running the Agent on a Sample Design**

The graph is invoked on one held-out test row as a demonstration, using the reference classical
model's own prediction as the numeric input the agent reasons over.

In [ ]:
sample_row = original_test.iloc[0]
sample_state: EngineDesignState = {
    "propellant": sample_row["propellant"],
    "chamber_pressure_MPa": sample_row["chamber_pressure_MPa"],
    "expansion_ratio": sample_row["expansion_ratio"],
    "mixture_ratio": sample_row["mixture_ratio"],
    "chamber_temp_K": sample_row["chamber_temp_K"],
    "exhaust_molar_mass": sample_row["exhaust_molar_mass"],
    "predicted_isp": float(classical_model.predict(X_test_final.iloc[[0]])[0]),
    "diagnosis": None,
    "recommendation": None,
}

result_state = engine_design_agent.invoke(sample_state)
print("Diagnosis:\n", result_state["diagnosis"])
print("\nRecommendation:\n", result_state["recommendation"])

### **16.3 - From Fixed Pipeline to Autonomous Agent**

The fixed pipeline above has a single hardcoded control flow: diagnose, then recommend, always in
that order, with no branching and no way for the LLM to ask for more information. It calls no
tools; the LLM only receives a formatted diagnosis string and writes prose in response. It
carries no memory between invocations; each call to `engine_design_agent.invoke` starts from a
blank state and produces a recommendation with no reference to any past review.

The agent built below removes those three constraints, on the three axes that define autonomy for
an agentic system: planning, tool use, and memory. Its control flow is a conditional edge: an
`agent` node calls the LLM with four tools bound to it, and a router function inspects the LLM's
response for tool calls, looping back to the `agent` node whenever a tool call is present and
moving to `END` once the LLM stops requesting tools, or once a small step cap is reached, which
guards against a free-tier model that repeats a tool call and never terminates on its own. The
LLM decides how many tool calls to make and in which order; no diagnosis logic is written in
Python if/elif branches this time. Its tools give it the ability to check physics metrics,
request a prediction, and look up precedent, each on its own initiative rather than as a step
handed to it by the graph. Its memory is a standing log, `data/agent_memory.jsonl`, written after
every run, so a later review of a similar design can cite the earlier one instead of starting
from first contact.

### **16.4 - Tools, Memory, and the ReAct Graph**

Four tools are bound to the LLM using the OpenAI-compatible tool-calling schema on the same HF
router client used elsewhere in this notebook. `compute_engine_metrics` and `predict_isp` give
the agent the same physics ratio and the same reference model prediction the fixed pipeline
computed for it in Section 16.1; the difference is that the agent now has to decide to ask for
them. `recall_similar_configs` reads the append-only memory log and returns the nearest past
cases by Euclidean distance on chamber pressure, expansion ratio, and mixture ratio, each
normalized to a comparable scale, with no vector database required at this scale of log.
`flag_for_engineering_review` is the terminal action the agent chooses to call once it judges the
review complete.

Tool-calling reliability on small open models varies. The 1.5B model used for the fixed
pipeline's prose generation is a light load for a single free-form completion, but chaining
several tool calls in sequence asks more of the model's instruction following. This section
upgrades to Qwen2.5-3B-Instruct for the autonomous agent specifically, a free-tier model still
small enough to run on the same HF router, chosen because more parameters generally make tool
call emission more reliable, and noted here explicitly rather than left as a silent swap.

In [ ]:
from typing import List

AGENT_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # upgraded from HF_MODEL for more reliable tool-call emission
MAX_AGENT_STEPS = 6
MEMORY_PATH = os.path.join(DATA_DIR, "agent_memory.jsonl")


def build_design_row(propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio):
    """Derive chamber temperature, exhaust molar mass, and the full feature row for one candidate
    design, reusing the physics and feature-engineering functions from Sections 7 through 9."""
    if propellant not in PROPELLANTS:
        raise ValueError(f"Unknown propellant. Choose from {list(PROPELLANTS.keys())}")
    params = PROPELLANTS[propellant]

    of_dev = (mixture_ratio - params["of_opt"]) / params["of_opt"]
    chamber_temp_K = params["tc_peak"] * (1 - 0.55 * of_dev**2)
    exhaust_molar_mass = params["m_peak"] * (1 + 0.12 * of_dev)
    gamma = params["gamma"]

    candidate_pe_pc = np.geomspace(1e-5, 0.5, 4000)
    candidate_epsilon = expansion_ratio_from_pressure_ratio(candidate_pe_pc, gamma)
    pressure_ratio_Pe_Pc = candidate_pe_pc[np.argmin(np.abs(candidate_epsilon - expansion_ratio))]

    row = pd.DataFrame([{
        "propellant": propellant,
        "chamber_pressure_MPa": chamber_pressure_MPa,
        "expansion_ratio": expansion_ratio,
        "mixture_ratio": mixture_ratio,
        "of_optimum": params["of_opt"],
        "chamber_temp_K": chamber_temp_K,
        "exhaust_molar_mass": exhaust_molar_mass,
        "gamma": gamma,
        "pressure_ratio_Pe_Pc": pressure_ratio_Pe_Pc,
    }])

    row = add_human_engineered_features(row)
    row, _ = build_ai_features(row, ai_feature_specs)
    return row, chamber_temp_K, exhaust_molar_mass


TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "compute_engine_metrics",
            "description": "Compute derived physics ratios (sqrt(Tc/M), mixture ratio deviation from "
                            "the propellant's optimum) for the engine design currently under review. "
                            "Takes no arguments.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "predict_isp",
            "description": "Predict the specific impulse (Isp, in seconds) of the engine design "
                            "currently under review, using the reference classical model from "
                            "Section 11. Takes no arguments.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "recall_similar_configs",
            "description": "Look up the most similar past engine designs logged in the agent's "
                            "memory file, with their predicted Isp and any past flag reason.",
            "parameters": {
                "type": "object",
                "properties": {
                    "n": {"type": "integer", "description": "Number of similar past cases to return."}
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "flag_for_engineering_review",
            "description": "Terminal action. Call this once a diagnosis is ready, with a short "
                            "reason, to close out the review.",
            "parameters": {
                "type": "object",
                "properties": {
                    "reason": {"type": "string", "description": "Short reason the design is being "
                                                                  "flagged for review."}
                },
                "required": ["reason"],
            },
        },
    },
]


def make_agent_tools(propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio):
    """Build the tool dispatch table and a mutable session record for one review, closing over the
    candidate design so the tool schemas above can stay argument-free where possible."""
    row, chamber_temp_K, exhaust_molar_mass = build_design_row(
        propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio
    )
    session_record = {
        "propellant": propellant,
        "chamber_pressure_MPa": chamber_pressure_MPa,
        "expansion_ratio": expansion_ratio,
        "mixture_ratio": mixture_ratio,
        "predicted_isp": None,
        "flag_reason": None,
    }

    def compute_engine_metrics():
        tc_over_m = (chamber_temp_K / exhaust_molar_mass) ** 0.5
        of_dev = (mixture_ratio - PROPELLANTS[propellant]["of_opt"]) / PROPELLANTS[propellant]["of_opt"]
        return (
            f"sqrt(Tc/M) = {tc_over_m:.3f}, the dominant Isp driver identified by permutation "
            f"importance in this notebook. Mixture ratio deviation from optimum = {of_dev:+.2%}. "
            f"Derived chamber temperature = {chamber_temp_K:.0f} K. Derived exhaust molar mass = "
            f"{exhaust_molar_mass:.1f} kg/kmol."
        )

    def predict_isp():
        predicted = float(classical_model.predict(row[FULL_FEATURES])[0])
        session_record["predicted_isp"] = predicted
        isp_range = PROPELLANT_ISP_RANGES.get(propellant, {"min": 0, "max": 500})
        return (
            f"Predicted Isp = {predicted:.1f} s. Typical {propellant} range is "
            f"{isp_range['min']:.0f} to {isp_range['max']:.0f} s."
        )

    def recall_similar_configs(n=3):
        if not os.path.exists(MEMORY_PATH):
            return "No past sessions logged yet."
        past = [json.loads(line) for line in open(MEMORY_PATH) if line.strip()]
        if not past:
            return "No past sessions logged yet."
        current = np.array([chamber_pressure_MPa, expansion_ratio, mixture_ratio])
        scale = np.array([10.0, 100.0, 3.0])  # rough per-parameter normalization
        scored = []
        for record in past:
            past_vec = np.array([
                record["chamber_pressure_MPa"], record["expansion_ratio"], record["mixture_ratio"]
            ])
            dist = float(np.linalg.norm((current - past_vec) / scale))
            scored.append((dist, record))
        scored.sort(key=lambda pair: pair[0])
        lines = []
        for dist, record in scored[:n]:
            has_isp = record["predicted_isp"] is not None
            isp_text = f"{record['predicted_isp']:.1f} s" if has_isp else "not computed"
            lines.append(
                f"- {record['propellant']} at Pc={record['chamber_pressure_MPa']:.1f} MPa, "
                f"eps={record['expansion_ratio']:.0f}, O/F={record['mixture_ratio']:.2f} "
                f"(distance {dist:.2f}): predicted Isp {isp_text}, flagged for: "
                f"{record['flag_reason'] or 'not flagged'}."
            )
        return "\n".join(lines)

    def flag_for_engineering_review(reason):
        session_record["flag_reason"] = reason
        return f"Flagged for engineering review: {reason}"

    dispatch = {
        "compute_engine_metrics": compute_engine_metrics,
        "predict_isp": predict_isp,
        "recall_similar_configs": recall_similar_configs,
        "flag_for_engineering_review": flag_for_engineering_review,
    }
    return dispatch, session_record


class AutonomousAgentState(TypedDict):
    messages: List[dict]
    step_count: int


def make_agent_node(dispatch):
    def agent_node(state):
        messages = state["messages"]
        step_count = state.get("step_count", 0) + 1
        if hf_client is None:
            fallback = {"role": "assistant", "content":
                        "No HF_TOKEN configured; the autonomous agent cannot run without a live LLM."}
            return {"messages": messages + [fallback], "step_count": step_count}
        try:
            response = hf_client.chat.completions.create(
                model=AGENT_MODEL, messages=messages, tools=TOOL_SCHEMAS, tool_choice="auto",
                max_tokens=350, temperature=0.2,
            )
            msg = response.choices[0].message
            new_message = {"role": "assistant", "content": msg.content or ""}
            if msg.tool_calls:
                new_message["tool_calls"] = [tc.model_dump() for tc in msg.tool_calls]
            return {"messages": messages + [new_message], "step_count": step_count}
        except Exception as exc:
            print("Autonomous agent LLM call failed, ending the loop. Error:", exc)
            fallback = {"role": "assistant", "content":
                        "HF router call failed; ending the review without a tool-driven diagnosis."}
            return {"messages": messages + [fallback], "step_count": step_count}
    return agent_node


def make_tools_node(dispatch):
    def tools_node(state):
        messages = state["messages"]
        last = messages[-1]
        tool_messages = []
        for call in last.get("tool_calls", []):
            name = call["function"]["name"]
            try:
                args = json.loads(call["function"]["arguments"] or "{}")
            except Exception:
                args = {}
            func = dispatch.get(name)
            if func is None:
                result = f"Unknown tool: {name}"
            else:
                try:
                    result = func(**args)
                except Exception as exc:
                    result = f"Tool {name} raised an error: {exc}"
            tool_messages.append({"role": "tool", "tool_call_id": call["id"], "content": str(result)})
        return {"messages": messages + tool_messages}
    return tools_node


def should_continue(state):
    last = state["messages"][-1]
    if last.get("tool_calls") and state.get("step_count", 0) < MAX_AGENT_STEPS:
        return "tools"
    return END


def build_autonomous_graph(dispatch):
    graph = StateGraph(AutonomousAgentState)
    graph.add_node("agent", make_agent_node(dispatch))
    graph.add_node("tools", make_tools_node(dispatch))
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")
    return graph.compile()


print("Autonomous agent graph builder ready. Tool model:", AGENT_MODEL)

### **16.5 - Running the Autonomous Agent on Sample Designs**

The wrapper below builds a fresh tool dispatch table for one candidate design, invokes the graph,
prints the tool-call trace so the planning behind the final recommendation is visible, and
appends the outcome to the memory log. It is run twice: once on a design with no precedent, and
once on a nearby design so `recall_similar_configs` has something to find.

In [ ]:
def run_autonomous_agent(propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio, verbose=True):
    dispatch, session_record = make_agent_tools(
        propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio
    )
    autonomous_agent = build_autonomous_graph(dispatch)

    system_message = {
        "role": "system",
        "content": (
            "You are an autonomous propulsion engineering review agent. You have tools to compute "
            "derived physics metrics, predict Isp, recall similar past designs, and flag a design "
            "for engineering review. Use as many tool calls as you need, in whatever order makes "
            "sense, before writing your final recommendation. Always call predict_isp at some point. "
            "Check recall_similar_configs so your recommendation can reference precedent if any "
            "exists. Call flag_for_engineering_review exactly once, with a short reason, as your "
            "last tool call, then write a short final recommendation with no further tool calls."
        ),
    }
    user_message = {
        "role": "user",
        "content": (
            f"Review this candidate engine design: propellant {propellant}, chamber pressure "
            f"{chamber_pressure_MPa} MPa, expansion ratio {expansion_ratio}, mixture ratio "
            f"{mixture_ratio}."
        ),
    }
    initial_state = {"messages": [system_message, user_message], "step_count": 0}

    try:
        final_state = autonomous_agent.invoke(initial_state, {"recursion_limit": 25})
    except Exception as exc:
        print("Autonomous agent run failed, error:", exc)
        return None

    if verbose:
        for message in final_state["messages"][2:]:
            role = message["role"]
            if role == "assistant" and message.get("tool_calls"):
                calls = ", ".join(tc["function"]["name"] for tc in message["tool_calls"])
                print(f"[agent] requests tool call(s): {calls}")
            elif role == "tool":
                print(f"[tool result] {message['content']}")
            elif role == "assistant":
                print(f"[agent] final recommendation: {message['content']}")

    if hf_client is not None:
        session_record["diagnosis"] = final_state["messages"][-1]["content"]
        with open(MEMORY_PATH, "a") as f:
            f.write(json.dumps(session_record) + "\n")

    return final_state


print("=== Autonomous agent run 1: no precedent yet ===")
run_autonomous_agent("LOX/RP-1", chamber_pressure_MPa=14.0, expansion_ratio=40, mixture_ratio=2.5)

print("\n=== Autonomous agent run 2: a nearby design, precedent now exists ===")
run_autonomous_agent("LOX/RP-1", chamber_pressure_MPa=14.5, expansion_ratio=42, mixture_ratio=2.55)

## **17 - Interactive Prediction Demo**

### **17.1 - Overview**

The function below ties the full pipeline together for a single candidate design. It accepts the
raw design parameters a propulsion engineer would set (propellant, chamber pressure, expansion
ratio, mixture ratio), derives the same human-engineered and AI-suggested features used in
training through the shared `build_design_row` helper from Section 16.4, predicts Isp with the
reference classical model, and runs a design recommendation agent. Chamber temperature and
exhaust molar mass are derived internally from the same mixture-ratio model used in Section 7, so
a user of this demo only needs to supply the four design parameters, not the combustion chemistry
outputs. The `use_autonomous_agent` flag chooses which of the two agentic capability tiers from
Section 16 handles the recommendation: the fixed diagnose-then-recommend pipeline by default, or
the ReAct-style autonomous agent with tools and memory when set to `True`.

In [ ]:
def predict_engine_design(propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio,
                           use_autonomous_agent=False):
    row, chamber_temp_K, exhaust_molar_mass = build_design_row(
        propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio
    )
    predicted_isp = float(classical_model.predict(row[FULL_FEATURES])[0])

    print(f"Propellant: {propellant}")
    print(f"Chamber pressure: {chamber_pressure_MPa} MPa | Expansion ratio: {expansion_ratio} "
          f"| Mixture ratio: {mixture_ratio}")
    print(f"Derived chamber temperature: {chamber_temp_K:.0f} K | Exhaust molar mass: "
          f"{exhaust_molar_mass:.1f} kg/kmol")
    print(f"\nPredicted Isp: {predicted_isp:.1f} s")

    if use_autonomous_agent:
        result = run_autonomous_agent(
            propellant, chamber_pressure_MPa, expansion_ratio, mixture_ratio, verbose=True
        )
        return result

    state: EngineDesignState = {
        "propellant": propellant,
        "chamber_pressure_MPa": chamber_pressure_MPa,
        "expansion_ratio": expansion_ratio,
        "mixture_ratio": mixture_ratio,
        "chamber_temp_K": chamber_temp_K,
        "exhaust_molar_mass": exhaust_molar_mass,
        "predicted_isp": predicted_isp,
        "diagnosis": None,
        "recommendation": None,
    }
    result = engine_design_agent.invoke(state)

    print(f"\nDiagnosis: {result['diagnosis']}")
    print(f"\nRecommendation: {result['recommendation']}")
    return result


_ = predict_engine_design("LOX/RP-1", chamber_pressure_MPa=14.0, expansion_ratio=40, mixture_ratio=2.5)

In [ ]:
# A second example, comparing a hydrogen upper-stage design against the hydrocarbon example above.
_ = predict_engine_design("LOX/LH2", chamber_pressure_MPa=18.0, expansion_ratio=90, mixture_ratio=5.9)

# A third example, routed through the autonomous agent from Section 16.3 through 16.5 instead of the
# fixed diagnose-then-recommend pipeline, so the same demo surfaces both agentic capability tiers.
_ = predict_engine_design("LOX/CH4", chamber_pressure_MPa=16.0, expansion_ratio=60, mixture_ratio=3.6,
                           use_autonomous_agent=True)

## **18 - Conclusion and Takeaways**

### **18.1 - Conclusion**

This notebook built a specific impulse prediction pipeline from a synthetic dataset grounded
directly in the ideal rocket nozzle equation, rather than from a black-box data source. The
pipeline combined hand-engineered propulsion ratios with LLM-suggested feature transforms, showed
that the combined feature set outperforms either source alone, and used an LLM-guided
augmentation strategy that kept the physics ground truth intact while filling in
under-represented regions of the design space. A gradient boosting model, a compact PyTorch
network, and the pretrained TabPFN foundation model were all evaluated on the same held-out Isp,
with TabPFN reaching a comparable fit using zero epochs of training on this dataset. Permutation
importance plus partial dependence analysis, chosen over SHAP to allow a clean comparison across
model families, recovered chamber temperature and exhaust molar mass as the dominant drivers of
Isp without ever being given their ratio directly. Plotting predictions against sqrt(Tc / M)
after the fact reproduced the same relationship that historically pushed propulsion engineers
toward liquid hydrogen for missions where propellant efficiency dominates the vehicle design, and
toward hydrocarbons and methane where density, storability, and reusability matter more. The
agentic layer then carried that prediction into two capability tiers: a fixed
diagnose-then-recommend pipeline, and a ReAct-style autonomous agent with tool use and a
persistent memory log that lets later reviews cite precedent.

### **18.2 - Takeaways**

- Isp depends on chamber temperature and exhaust molecular weight only through the ratio Tc / M,
  under a square root. A model given the two raw quantities separately can still recover this
  relationship through permutation importance and partial dependence analysis, without ever seeing
  the ratio as a feature.
- Hydrogen's advantage over hydrocarbon and storable propellants comes almost entirely from its low
  exhaust molecular weight, not from unusually high chamber temperature. Chamber temperature differs
  by only a few percent across the propellant families in this dataset; molecular weight differs by
  nearly a factor of two.
- LLM-suggested features and hand-engineered physics features are complementary rather than
  redundant, and an ablation study is the correct way to check that before committing to either source
  alone.
- Using a language model to propose sampling distributions for under-represented design regions,
  while keeping the actual data generation inside a physics function, is a defensible way to use GenAI
  for synthetic augmentation without contaminating the ground truth. The three-way generalization
  check against a fixed original holdout set is what makes that defensibility verifiable rather than
  assumed.
- Permutation importance and partial dependence plots are a practical alternative to SHAP when the
  goal is comparing two structurally different models on the same footing, since both methods depend
  only on a model's predict function and its performance on held-out data.
- A pretrained tabular foundation model is a meaningfully different capability tier from a custom-trained model. TabPFN reaches a comparable R-squared to the gradient boosting model and the feedforward network with no per-dataset training at all, on a dataset small and structured enough to sit inside its documented operating range.
- Planning, tool use, and memory are the three axes that separate an agentic pipeline from an autonomous agent. The fixed diagnose-then-recommend graph has none of the three; the ReAct-style agent has all three, and its recommendations can cite precedent from a persistent memory log that the fixed pipeline has no way to keep.